# Bring Your Own Model with Amazon SageMaker AI: Script Mode in SDK v3

This notebook walks through two end-to-end examples demonstrating **script mode** in the SageMaker Python SDK v3:

1. **Example 1** — Train and deploy a scikit-learn Random Forest (tabular ML → real-time endpoint)
2. **Example 2** — Fine-tune Stable Diffusion 3.5 with LoRA (generative AI, multi-GPU distributed training)

Both examples use the same two core classes:
- `ModelTrainer` — configures and launches a SageMaker training job
- `ModelBuilder` — packages your inference handler and deploys to an endpoint

**Key concept:** The `SourceCode` object syncs a local code directory into the container at runtime — your code runs inside the container without being baked into the image.

---
## Prerequisites

- An AWS account with Amazon SageMaker AI access
- An IAM execution role with SageMaker and S3 permissions
- SageMaker Python SDK v3 installed
- Training container images pushed to Amazon ECR (see container build sections below)
- An S3 bucket for training data and model artifacts
- (Optional) An MLflow tracking server on SageMaker for experiment tracking

In [ ]:
# Install/upgrade dependencies
!pip install "sagemaker>=3.0" boto3 mlflow sagemaker-mlflow

---
## Container Setup (Prerequisites)

Before running the training examples, you need to build and push Docker images to ECR.
These containers are intentionally minimal — they contain only runtime/framework libraries,
**no training code**. All algorithm-specific code is injected at runtime via `SourceCode`.

### Scikit-learn Container

**Dockerfile** (`docker/sklearn/Dockerfile.sklearn`):
```dockerfile
FROM python:3.13-slim

RUN apt-get update && apt-get install -y \
    build-essential jq git \
    && rm -rf /var/lib/apt/lists/*

COPY docker/sklearn/requirements.txt .
RUN pip install -r requirements.txt --no-cache-dir
```

**requirements.txt:**
```
scikit-learn>=1.8.0
pandas>=2.0.0
matplotlib>=3.6.1
mlflow>=3.10.0
sagemaker-mlflow>=0.2.0
```

### Stable Diffusion Container

**Dockerfile** (`docker/stable_diffusion/Dockerfile.stablediffusion`):
```dockerfile
FROM pytorch/pytorch:2.7.1-cuda12.8-cudnn9-devel

RUN apt-get update && apt-get install -y \
    build-essential jq git \
    && rm -rf /var/lib/apt/lists/*

COPY docker/pytorch/requirements.txt .
RUN pip install -r requirements.txt --no-cache-dir
```

### Build & Push to ECR

Run these from a terminal in the project root (requires Docker installed locally):

```bash
# Sklearn container
bash build.sh --env .env.docker.sklearn
bash push.sh --env .env.docker.sklearn

# Stable Diffusion container
bash build.sh --env .env.docker.stablediffusion
bash push.sh --env .env.docker.stablediffusion
```

The `build.sh` and `push.sh` scripts handle ECR authentication, repository creation, tagging, and pushing automatically.

---
# Example 1: Train and Deploy a Scikit-learn Model

We train a Random Forest classifier on the diabetes dataset and deploy it to a real-time SageMaker endpoint.

## Step 1: Configuration

In [ ]:
import os

import boto3
from sagemaker.core.helper.session_helper import Session
from sagemaker.core.training.configs import (
    Compute,
    InputData,
    OutputDataConfig,
    SourceCode,
    StoppingCondition,
)
from sagemaker.serve.model_builder import ModelBuilder
from sagemaker.serve.mode.function_pointers import Mode
from sagemaker.serve.utils.types import ModelServer
from sagemaker.train.model_trainer import ModelTrainer

# ============================================================
# Auto-detect account context
# ============================================================
boto_session = boto3.Session()
sm_session = Session(boto_session=boto_session)

AWS_REGION = boto_session.region_name
AWS_ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
SAGEMAKER_EXECUTION_ROLE = sm_session.get_caller_identity_arn()  # works for role or user
S3_BUCKET = sm_session.default_bucket()

print(f"Account : {AWS_ACCOUNT_ID}")
print(f"Region  : {AWS_REGION}")
print(f"Role    : {SAGEMAKER_EXECUTION_ROLE}")
print(f"Bucket  : {S3_BUCKET}")

In [ ]:
# ============================================================
# Training configuration
# ============================================================

# Your custom training image pushed to ECR (built in the Container Setup step
# above). Alternatively, you can point this at a prebuilt AWS Deep Learning
# Container instead of building your own — see the full list of training and
# inference DLC image URIs at:
# https://aws.github.io/deep-learning-containers/reference/available_images/
TRAINING_IMAGE_URI = f"{AWS_ACCOUNT_ID}.dkr.ecr.{AWS_REGION}.amazonaws.com/sklearn:latest"

# S3 path
MODEL_OUTPUT_S3_PATH = f"s3://{S3_BUCKET}/random-forest/model-output"

# MLflow on SageMaker (set to None to skip experiment tracking)
MLFLOW_ARN = "arn:aws:sagemaker:<your-region>:<your-account-id>:mlflow-app/<your-mlflow-app-id>"  # Replace with your MLflow tracking server ARN, or set to None to skip
MLFLOW_EXPERIMENT_NAME = "random-forest-experiment"

# Job name
JOB_NAME = "random-forest-training"

## Step 2: Launch a Training Job with ModelTrainer

The `SourceCode` object takes your local `source_dir` and a `command` string. At job launch, SageMaker syncs the entire `source_dir` into the container and runs your command. This decouples your code from your container image — change the script, re-launch. No container rebuild needed.

In [ ]:
source_code = SourceCode(
    source_dir="./train/random_forest",
    command=(
        "python random_forest.py"
        " --n_jobs 4 --max_depth 10 --n_estimators 120"
        f" --mlflow_arn {MLFLOW_ARN}"
        f" --mlflow_experiment_name {MLFLOW_EXPERIMENT_NAME}"
    ),
)

compute = Compute(
    instance_type="ml.m5.2xlarge",
    instance_count=1,
    volume_size_in_gb=30,
    keep_alive_period_in_seconds=3600,  # Warm pool for faster re-runs
)

stopping_condition = StoppingCondition(max_runtime_in_seconds=3600)
output_config = OutputDataConfig(s3_output_path=MODEL_OUTPUT_S3_PATH)

# Note: no input_data_config — the training script fetches the Pima Indians
# Diabetes dataset directly from OpenML at runtime, so no S3 input channel
# is needed.
model_trainer = ModelTrainer(
    training_image=TRAINING_IMAGE_URI,
    source_code=source_code,
    compute=compute,
    output_data_config=output_config,
    stopping_condition=stopping_condition,
    role=SAGEMAKER_EXECUTION_ROLE,
    base_job_name=JOB_NAME,
    sagemaker_session=Session(),
)

print(f"Starting training job: {JOB_NAME}")
model_trainer.train(wait=True)
training_job_name = model_trainer._latest_training_job.training_job_name
print(f"Training complete: {training_job_name}")

## Step 3: Locate the Trained Model Artifact

In [ ]:
sm_session = Session()
sm_client = sm_session.boto_session.client("sagemaker")
training_job_desc = sm_client.describe_training_job(TrainingJobName=training_job_name)
model_artifact_s3_uri = training_job_desc["ModelArtifacts"]["S3ModelArtifacts"]
print(f"Model artifact: {model_artifact_s3_uri}")

## Step 4: Deploy to a Real-time Endpoint with ModelBuilder

ModelBuilder packages your inference handler, repacks it with the model artifact, and creates the endpoint. The same `SourceCode` pattern is used: point at a local directory containing your handler and specify the `entry_script`.

In [ ]:
# ============================================================
# Inference configuration
# ============================================================
# For deployment we use a prebuilt AWS Deep Learning Container (the DJL
# Serving inference image) — no build needed. The URI is region-dynamic.
# Browse all prebuilt training and inference DLC images here:
# https://aws.github.io/deep-learning-containers/reference/available_images/
INFERENCE_IMAGE_URI = (
    f"763104351884.dkr.ecr.{AWS_REGION}.amazonaws.com/djl-inference:0.36.0-cpu-full"
)
ENDPOINT_NAME = "random-forest-endpoint"

inference_source_code = SourceCode(
    source_dir="./deploy/random_forest",
    entry_script="inference.py",
)

model_builder = ModelBuilder(
    image_uri=INFERENCE_IMAGE_URI,
    model_server=ModelServer.DJL_SERVING,
    source_code=inference_source_code,
    s3_model_data_url=model_artifact_s3_uri,
    # DJL Serving routes to our handler via OPTION_ENTRYPOINT. Without it, DJL
    # falls back to its built-in sklearn_handler (which expects a .skops file)
    # and fails to load our model.joblib — so this env var is required.
    env_vars={"OPTION_ENTRYPOINT": "code/inference.py"},
    sagemaker_session=Session(),
    role_arn=SAGEMAKER_EXECUTION_ROLE,
)

# Workaround for v3 SDK quirk
model_builder.model_path = "/tmp/sagemaker/model-builder"
os.makedirs(model_builder.model_path, exist_ok=True)
model_builder.dependencies = []

print(f"Building model package: {ENDPOINT_NAME}")
model_builder.build(model_name=ENDPOINT_NAME, mode=Mode.SAGEMAKER_ENDPOINT)

print(f"Deploying endpoint: {ENDPOINT_NAME}")
predictor = model_builder.deploy(
    endpoint_name=ENDPOINT_NAME,
    initial_instance_count=1,
)
print(f"Endpoint ready: {ENDPOINT_NAME}")

## Step 5: Test the Endpoint

In [ ]:
# Pima Indians Diabetes dataset: 8 numeric features
# (preg, plas, pres, skin, insu, mass, pedi, age)
sample_csv = "6,148,72,35,0,33.6,0.627,50"

# v3: model_builder.deploy() returns a sagemaker.core.resources.Endpoint
# object — call .invoke() directly instead of using a boto3 runtime client.
response = predictor.invoke(
    body=sample_csv.encode("utf-8"),
    content_type="text/csv",
)

# response is an InvokeEndpointOutput; .body is a botocore StreamingBody
print("Prediction:", response.body.read().decode("utf-8"))

# The model returns a class label: 1 = tested_positive (diabetes), 0 = tested_negative.
# This sample is the first row of the Pima dataset (true label: positive), so a prediction
# of [1] confirms the endpoint is serving the trained classifier correctly.

---
# Example 2: Fine-tune Stable Diffusion 3.5 with LoRA

This example demonstrates the same `ModelTrainer` + `SourceCode` pattern for a sophisticated generative AI training job. We fine-tune Stable Diffusion 3.5 Medium using LoRA with Hugging Face Accelerate for multi-GPU distributed training on `ml.g5.12xlarge` (4× A10G GPUs).

The source directory structure:
```
train/stable_diffusion/
├── base.sh                         # Launcher: pip installs, GPU detection, accelerate launch
├── train_text_to_image_lora.py     # Main training script
├── requirements.txt                # Runtime deps installed by base.sh
├── recipes/
│   └── default-medium-g5_12x.yaml  # Hyperparameter recipe
└── accelerate_configs/
    └── ddp.yaml                    # Distributed training config
```

Swap recipes, adjust LoRA rank, or change the base model ID with a one-line YAML edit — no Docker rebuild required.

## Step 1: Configuration

### Managing secrets with AWS Secrets Manager

This example needs two secrets at runtime:

- **`HF_TOKEN`** — a Hugging Face token with access to the gated **Stable Diffusion 3.5 Medium** repo (required for the model download).
- **`WANDB_API_KEY`** — a Weights & Biases API key (optional, for experiment logging).

Rather than pasting these tokens into the notebook (where they'd end up in the job definition and CloudTrail), we store them once in **AWS Secrets Manager** and pass only the secret's **ARN** to the training job. `base.sh` fetches the secret at runtime and exports each key as an environment variable inside the container — so the raw tokens never leave Secrets Manager.

**Set it up with the included CloudFormation template.** From the project root, deploy the template in `cloudformation/training-secrets.yaml`, passing your real token values:

```bash
aws cloudformation deploy \
  --template-file cloudformation/training-secrets.yaml \
  --stack-name script-mode-blog-secrets \
  --parameter-overrides \
      HuggingFaceToken=hf_your_real_token_here \
      WandbApiKey=your_wandb_key_here \
  --capabilities CAPABILITY_IAM \
  --region us-east-2
```

The stack creates the secret (a JSON object `{"HF_TOKEN": "...", "WANDB_API_KEY": "..."}`) and an IAM managed policy that grants read access to it. Retrieve the two outputs with:

```bash
aws cloudformation describe-stacks \
  --stack-name script-mode-blog-secrets \
  --region us-east-2 \
  --query "Stacks[0].Outputs" --output table
```

Then:

1. **Attach** the `SecretReadPolicyArn` managed policy to your **SageMaker execution role** so the training job can read the secret. (If your role uses `AmazonSageMakerFullAccess`, it can already read secrets whose names start with `AmazonSageMaker-`, which this one does.)
2. **Copy** the `SecretArn` output value into the `SECRETS_ARN` variable in the configuration cell below.

In [ ]:
# ============================================================
# EDIT THESE VALUES FOR YOUR ACCOUNT
# ============================================================
SD_TRAINING_IMAGE_URI = f"{AWS_ACCOUNT_ID}.dkr.ecr.{AWS_REGION}.amazonaws.com/stablediffusion:latest"

SD_S3_BUCKET = S3_BUCKET  # Reuse the same bucket or set a different one
SD_TRAINING_DATA_S3_PATH = f"s3://{SD_S3_BUCKET}/stable-diffusion/data"
SD_MODEL_OUTPUT_S3_PATH = f"s3://{SD_S3_BUCKET}/stable-diffusion/model-output"

SD_JOB_NAME = "stable-diffusion-training"

# MLflow on SageMaker (set SD_MLFLOW_ARN to None to skip experiment tracking).
# Passed through base.sh to the training script, which points MLflow at this
# tracking server before Accelerate starts logging (same pattern as Example 1).
SD_MLFLOW_ARN = "arn:aws:sagemaker:<your-region>:<your-account-id>:mlflow-app/<your-mlflow-app-id>"  # Replace with your MLflow tracking server ARN, or set to None to skip
SD_MLFLOW_EXPERIMENT_NAME = "stable-diffusion-experiment"

# Secrets via AWS Secrets Manager (see the markdown cell above for setup).
# Pass the ARN of a secret holding a JSON object with the keys HF_TOKEN and
# WANDB_API_KEY. The training job fetches it at runtime and exports each key as
# an environment variable inside the container — so the gated SD 3.5 download
# (HF_TOKEN) and W&B logging (WANDB_API_KEY) work without the raw tokens ever
# appearing in this notebook, the job definition, or CloudTrail.
#
# SD 3.5 Medium is a gated HF repo, so the HF_TOKEN stored in the secret must be
# a real token with access granted (a dummy value will fail the model download).
SECRETS_ARN = "arn:aws:secretsmanager:<your-region>:<your-account-id>:secret:AmazonSageMaker-script-mode-blog-secrets-XXXXXX"  # Replace with the SecretArn output from the CloudFormation stack

## Step 2: Launch Training Job

The `command` launches `base.sh` — a bash launcher that detects the GPU count, runs `accelerate launch`, and kicks off LoRA fine-tuning. All hyperparameters live in the YAML recipe file.

In [ ]:
sd_source_code = SourceCode(
    source_dir="./train/stable_diffusion",
    command=(
        "/bin/bash base.sh"
        " --config recipes/default-medium-g5_12x.yaml"
        " --training-script train_text_to_image_lora.py"
        " --accelerate-config accelerate_configs/ddp.yaml"
        f" --mlflow-arn {SD_MLFLOW_ARN}"
        f" --mlflow-experiment-name {SD_MLFLOW_EXPERIMENT_NAME}"
    ),
)

sd_compute = Compute(
    instance_type="ml.g5.12xlarge",  # 4x A10G GPUs
    instance_count=1,
    volume_size_in_gb=30,
    keep_alive_period_in_seconds=3600,
)

sd_stopping_condition = StoppingCondition(max_runtime_in_seconds=24 * 3600)
sd_output_config = OutputDataConfig(s3_output_path=SD_MODEL_OUTPUT_S3_PATH)

sd_input_config = [
    InputData(channel_name="train", data_source=SD_TRAINING_DATA_S3_PATH),
]

# We pass the Secrets Manager ARN (not the raw tokens) via `environment`.
# base.sh reads SECRETS_ARN, fetches the secret, and exports HF_TOKEN and
# WANDB_API_KEY as environment variables before launching training.
sd_model_trainer = ModelTrainer(
    training_image=SD_TRAINING_IMAGE_URI,
    source_code=sd_source_code,
    compute=sd_compute,
    output_data_config=sd_output_config,
    stopping_condition=sd_stopping_condition,
    role=SAGEMAKER_EXECUTION_ROLE,
    environment={"SECRETS_ARN": SECRETS_ARN},
    base_job_name=SD_JOB_NAME,
    sagemaker_session=Session(),
)

print(f"Starting training job: {SD_JOB_NAME}")
sd_model_trainer.train(input_data_config=sd_input_config, wait=True)
sd_training_job_name = sd_model_trainer._latest_training_job.training_job_name
print(f"Training complete: {sd_training_job_name}")

## Step 3: Locate the Trained LoRA Weights

In [ ]:
sd_sm_session = Session()
sd_sm_client = sd_sm_session.boto_session.client("sagemaker")
sd_training_job_desc = sd_sm_client.describe_training_job(TrainingJobName=sd_training_job_name)
sd_model_artifact_s3_uri = sd_training_job_desc["ModelArtifacts"]["S3ModelArtifacts"]
print(f"LoRA weights: {sd_model_artifact_s3_uri}")

## Step 4: Deploy the Fine-tuned Model to a Real-time Endpoint

We deploy with the same `ModelBuilder` + `SourceCode` pattern as Example 1, but for a GPU generative model. Two differences from the scikit-learn endpoint:

1. **GPU inference image** — we use a DJL LMI (Large Model Inference) container with CUDA, since Stable Diffusion needs a GPU.
2. **Secrets at inference time** — the handler loads the *base* SD 3.5 model from Hugging Face at startup (a gated repo), then applies our trained LoRA weights on top. So the endpoint also needs `HF_TOKEN` — we pass the same `SECRETS_ARN` via `env_vars`, and `inference.py` fetches it exactly like `base.sh` did during training.

As in Example 1, we set `OPTION_ENTRYPOINT` so DJL Serving routes requests to our `code/inference.py` handler instead of a built-in one.

In [ ]:
# ============================================================
# Inference configuration (Stable Diffusion)
# ============================================================
# GPU DJL LMI inference container. Browse all prebuilt DLC image URIs at:
# https://aws.github.io/deep-learning-containers/reference/available_images/
SD_INFERENCE_IMAGE_URI = (
    f"763104351884.dkr.ecr.{AWS_REGION}.amazonaws.com/djl-inference:0.36.0-lmi27.0.0-cu130"
)
SD_ENDPOINT_NAME = "stable-diffusion-endpoint"

# Same SourceCode pattern as Example 1 — point at the handler directory.
sd_inference_source_code = SourceCode(
    source_dir="./deploy/stable_diffusion",
    entry_script="inference.py",
)

sd_model_builder = ModelBuilder(
    image_uri=SD_INFERENCE_IMAGE_URI,
    model_server=ModelServer.DJL_SERVING,
    source_code=sd_inference_source_code,
    # The training job (base.sh) staged serving-requirements.txt into the model
    # artifact root, so this model.tar.gz already contains a root-level
    # requirements.txt. The DJL container pip-installs it at startup — that's how
    # inference.py gets boto3 (Secrets Manager) + diffusers (to load SD 3.5 + LoRA).
    s3_model_data_url=sd_model_artifact_s3_uri,
    instance_type="ml.g5.4xlarge",  # 1x A10G (24GB) — SD needs a GPU; set here (deploy() reads self.instance_type)
    # DJL routing via env vars (SourceCode packs the handler under code/):
    #   OPTION_ENGINE=Python  -> Python engine (else inferEngine fails to detect one)
    #   OPTION_ENTRYPOINT     -> our handler under code/
    #   SECRETS_ARN           -> inference.py fetches HF_TOKEN for the gated base model
    env_vars={
        "OPTION_ENGINE": "Python",
        "OPTION_ENTRYPOINT": "code/inference.py",
        "SECRETS_ARN": SECRETS_ARN,
    },
    sagemaker_session=Session(),
    role_arn=SAGEMAKER_EXECUTION_ROLE,
)

# Workaround for v3 SDK quirk (same as Example 1)
sd_model_builder.model_path = "/tmp/sagemaker/sd-model-builder"
os.makedirs(sd_model_builder.model_path, exist_ok=True)
sd_model_builder.dependencies = []

print(f"Building model package: {SD_ENDPOINT_NAME}")
sd_model_builder.build(model_name=SD_ENDPOINT_NAME, mode=Mode.SAGEMAKER_ENDPOINT)

# Guard against build()'s instance auto-detect overriding our choice.
sd_model_builder.instance_type = "ml.g5.4xlarge"

print(f"Deploying endpoint (this can take several minutes): {SD_ENDPOINT_NAME}")
sd_predictor = sd_model_builder.deploy(
    endpoint_name=SD_ENDPOINT_NAME,
    initial_instance_count=1,
)
print(f"Endpoint ready: {SD_ENDPOINT_NAME}")

## Step 5: Test the Endpoint

Send a JSON prompt and display the generated image. The handler returns a base64-encoded PNG, which we decode and render inline.

In [ ]:
import base64
import io
import json

from PIL import Image

# The prompt we fine-tuned on (see the recipe's validation_prompt)
payload = {
    "prompt": "a boy Malcom and his dog Ben",
    "num_inference_steps": 30,
    "guidance_scale": 7.5,
    "seed": 42,
}

# v3: sd_model_builder.deploy() returns an Endpoint object — call .invoke() directly.
response = sd_predictor.invoke(
    body=json.dumps(payload).encode("utf-8"),
    content_type="application/json",
)

result = json.loads(response.body.read().decode("utf-8"))

# Decode the base64 PNG the handler returned and display it
image = Image.open(io.BytesIO(base64.b64decode(result["generated_image"])))
print(f"Prompt: {result['prompt']}")
print(f"Parameters: {result['parameters']}")
image  # renders inline in the notebook

---
# Clean Up

Delete resources to avoid ongoing charges.

In [ ]:
from sagemaker.core.resources import Endpoint, EndpointConfig, Model

# Delete each endpoint, its endpoint config, and the model using the SDK v3
# resource classes: fetch each resource with .get(), then .delete().
# (ModelBuilder names all three after the endpoint name.)
for name in [ENDPOINT_NAME, SD_ENDPOINT_NAME]:
    try:
        Endpoint.get(endpoint_name=name).delete()
        EndpointConfig.get(endpoint_config_name=name).delete()
        Model.get(model_name=name).delete()
        print(f"Deleted endpoint: {name}")
    except Exception as e:
        print(f"Skipping {name}: {e}")

print("\nRemember to also clean up:")
print("  - S3 training data and model artifacts (if no longer needed)")
print("  - ECR container images (if no longer needed)")
print("  - MLflow tracking server (if created for this walkthrough)")
print("  - The Secrets Manager stack (script-mode-blog-secrets) if no longer needed")